In [1]:
import tqdm
import os
import numpy as np
import matplotlib.pyplot as plta
import spikeinterface.full as si
import spikeinterface.sorters as ss 
import scipy.signal
from tqdm.auto import tqdm, trange
import urllib.request


The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [3]:
root_folder = '/u/gongt/GLC7189_g_g1/' 
data_folder = os.path.join(root_folder, 'preprocessed')
output_folder_name = "kilosort4_nblock1_raventest"

rec = si.load_extractor(data_folder)
print(rec)

BinaryFolderRecording: 368 channels - 30.0kHz - 1 segments - 1,082,222,502 samples 
                       36,074.08s (10.02 hours) - float16 dtype - 741.81 GiB


/tmp/ipykernel_135875/35109639.py:5: DeprecationWarning: load_extractor() is deprecated and will be removed in version 0.104.0. Please use load() instead.
  rec = si.load_extractor(data_folder)


In [6]:
fs = rec.get_sampling_frequency()
samples_to_keep = int(3600 * fs)

rec_trimmed = rec.frame_slice(start_frame=0, end_frame=samples_to_keep)
rec_trimmed = rec_trimmed.astype(np.float32)
print(rec_trimmed)

astype (AstypeRecording): 368 channels - 30.0kHz - 1 segments - 108,000,000 samples 
        3,600.00s (60.00 minutes) - float32 dtype - 148.06 GiB


In [5]:
ks_params = si.get_default_sorter_params('kilosort4')
print(ks_params)

{'batch_size': 60000, 'nblocks': 1, 'Th_universal': 9, 'Th_learned': 8, 'nt': 61, 'shift': None, 'scale': None, 'artifact_threshold': inf, 'nskip': 25, 'whitening_range': 32, 'highpass_cutoff': 300, 'binning_depth': 5, 'sig_interp': 20, 'drift_smoothing': [0.5, 0.5, 0.5], 'nt0min': None, 'dmin': None, 'dminx': 32, 'min_template_size': 10, 'template_sizes': 5, 'nearest_chans': 10, 'nearest_templates': 100, 'max_channel_distance': 32, 'max_peels': 100, 'templates_from_data': True, 'n_templates': 6, 'n_pcs': 6, 'Th_single_ch': 6, 'acg_threshold': 0.2, 'ccg_threshold': 0.25, 'cluster_neighbors': 10, 'cluster_downsampling': 1, 'max_cluster_subset': 25000, 'x_centers': None, 'duplicate_spike_ms': 0.25, 'position_limit': 100, 'do_CAR': True, 'invert_sign': False, 'save_extra_vars': False, 'save_preprocessed_copy': False, 'torch_device': 'auto', 'bad_channels': None, 'clear_cache': False, 'do_correction': True, 'skip_kilosort_preprocessing': False, 'keep_good_only': False, 'use_binary_file': T

In [6]:
ks_params = si.get_default_sorter_params('kilosort4')
ks_params['do_CAR'] = False # so as not to common reference twice

# comprises highpass filter (>300 Hz) and whitening (right? see https://kilosort.readthedocs.io/en/latest/api.html#kilosort.run_kilosort.compute_preprocessing)
ks_params['skip_kilosort_preprocessing'] = False
ks_params['do_correction'] = True
ks_params['nblocks'] = 0
ks_params['n_pcs'] = 12
ks_params['ccg_threshold'] = 0.1

from datetime import datetime

output_folder = f"root_folder/{output_folder_name}"
params_file = os.path.join(output_folder, "kilosort_params.txt")

if os.path.exists(output_folder):
    import shutil
    shutil.rmtree(output_folder) 

sorting = si.run_sorter(
    'kilosort4',
    rec,
    folder=output_folder,
    docker_image=False,
    verbose=True,
    **ks_params,
)

with open(params_file, "w") as f:
    f.write(f"Kilosort parameters saved on {datetime.now()}\n")
    f.write(f"SpikeInterface version: {si.__version__}\n\n")
    f.write("Parameters:\n")
    for key, value in ks_params.items():
        f.write(f"{key}: {value}\n")
print(f"Full Kilosort parameters saved to: {params_file}")

kilosort.run_kilosort:  
kilosort.run_kilosort: Computing preprocessing variables.
kilosort.run_kilosort: ----------------------------------------


Skipping common average reference.


/u/gongt/.local/lib/python3.10/site-packages/kilosort/io.py:631: RuntimeWarning: 
                float16 is not supported and may result in unexpected
                behavior or errors. Supported types are:

                ['int16', 'uint16', 'int32', 'float32']
                
  warnings.warn(message, RuntimeWarning)
kilosort.run_kilosort: N samples: 1082222502
kilosort.run_kilosort: N seconds: 36074.0834
kilosort.run_kilosort: N batches: 18038


KeyboardInterrupt: 

In [ ]:
import spikeinterface as si
import spikeinterface.sorters as ss

print(f"SpikeInterface: {si.__version__}")
print("Available sorters:", ss.available_sorters())

# Try to import kilosort directly
try:
    import kilosort
    print(f"Kilosort: {kilosort.__version__}")
    print(f"Kilosort path: {kilosort.__file__}")
except ImportError as e:
    print(f"Kilosort import error: {e}")

SpikeInterface: 0.103.1
Available sorters: ['combinato', 'hdsort', 'herdingspikes', 'ironclust', 'kilosort', 'kilosort2', 'kilosort2_5', 'kilosort3', 'kilosort4', 'mountainsort4', 'mountainsort5', 'pykilosort', 'rtsort', 'simple', 'spykingcircus', 'spykingcircus2', 'tridesclous', 'tridesclous2', 'waveclus', 'waveclus_snippets', 'yass']
Kilosort: 4.1.2
Kilosort path: /u/gongt/.local/lib/python3.10/site-packages/kilosort/__init__.py
